In [2]:
############################################## 
# 1. Data Preprocessing 
############################################## 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


df = pd.read_csv('Clean_Dataset.csv')

# 과적합을 유발하는 편명(flight)과 불필요한 인덱스 삭제
columns_to_drop = []
if 'Unnamed: 0' in df.columns:
    columns_to_drop.append('Unnamed: 0')
if 'flight' in df.columns:
    columns_to_drop.append('flight')
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# 변수 생성 (Feature Engineering)
# 노선(Route) 변수 생성: 출발지_도착지, 중복 제거 
df['route'] = df['source_city'] + "_" + df['destination_city']
df.drop(columns=['source_city', 'destination_city'], inplace=True)

# 예약 시점 구간 변수 생성
# -1부터 시작하여 0일 데이터 손실(결측치) 완벽 방지
bins = [-1, 7, 21, df['days_left'].max()]
labels = ['1~7일 전(출발 직전)', '8~21일 전(단기)', '22일 이상(장기)']
df['booking_period'] = pd.cut(df['days_left'], bins=bins, labels=labels)

# 타겟 변수 로그 변환 (가격 변동성 학습 최적화)
df['price'] = np.log1p(df['price'])

# ============================================
# 1-2. Encoding
# ============================================

# class (0과 1로 매핑)
df['class'] = df['class'].map({'Economy': 0, 'Business': 1})

# stops: 직항 0, 1회 경유 1, 2회 이상 2
df['stops'] = df['stops'].map({'zero': 0, 'one': 1, 'two_or_more': 2})

# 문자열 데이터 원-핫 인코딩
df_encoded = pd.get_dummies(df, drop_first=True)

# ================================================
# 1-3. Train/Test Split - 8:2
# ================================================
X = df_encoded.drop(columns=['price'])
y = df_encoded['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("=== 최종 전처리 및 데이터 분리 완료 ===")
print(f"X_train 데이터 크기: {X_train.shape}")
print(f"X_test 데이터 크기: {X_test.shape}")
print(f"y_train 데이터 크기: {y_train.shape}")
print(f"y_test 데이터 크기: {y_test.shape}")

# 전체 전처리 완료된 데이터 하나로 저장하기
df.to_csv('final_preprocessed_data.csv', index=False, encoding='utf-8-sig')

# Train / Test 데이터를 각각 저장하기 (y컬럼을 다시 붙여서 )

train_df = pd.concat([X_train, y_train], axis=1)
train_df.to_csv('train_data.csv', index=False, encoding='utf-8-sig')

test_df = pd.concat([X_test, y_test], axis=1)
test_df.to_csv('test_data.csv', index=False, encoding='utf-8-sig')

print("CSV 파일 저장이 완료되었습니다. 폴더를 확인해 주세요.")


=== 최종 전처리 및 데이터 분리 완료 ===
X_train 데이터 크기: (240122, 50)
X_test 데이터 크기: (60031, 50)
y_train 데이터 크기: (240122,)
y_test 데이터 크기: (60031,)
CSV 파일 저장이 완료되었습니다. 폴더를 확인해 주세요.


In [ ]:
############################################## 
# 1. Ridge
############################################## 

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.linear_model import Ridge, RidgeCV                                       
import numpy as np                                    


# [Economy 데이터 분할]
df_eco = df_encoded[df_encoded['class'] == 0].copy()
X_eco = df_eco.drop(columns=['price'])
y_eco = df_eco['price']
X_train_eco, X_test_eco, y_train_eco, y_test_eco = train_test_split(X_eco, y_eco, test_size=0.2, random_state=42)

# [Business 데이터 분할]
df_bus = df_encoded[df_encoded['class'] == 1].copy()
X_bus = df_bus.drop(columns=['price'])
y_bus = df_bus['price']
X_train_bus, X_test_bus, y_train_bus, y_test_bus = train_test_split(X_bus, y_bus, test_size=0.2, random_state=42)


# =====================================================================
# 1. 클래스별 데이터 분리 및 학습 함수
# =====================================================================

def analyze_ridge_log_scale(X_train, X_test, y_train, y_test, class_name):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    # 2. 모델 학습
    model = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], scoring='r2')
    model.fit(X_train_s, y_train)
    
    # 3. 예측 및 평가
    ridge_y_pred = model.predict(X_test_s)
  
    ridge_train_score = model.score(X_train_s, y_train) 
    ridge_test_score = model.score(X_test_s, y_test)   
    ridge_mae = mean_absolute_error(y_test, ridge_y_pred)
    ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_y_pred))
    
    print(f"\n--- [{class_name}] Ridge 결과 ---")

    print(f"최적 Alpha 값: {model.alpha_}")
    print(f"Train R2 Score: {ridge_train_score:.4f}") 
    print(f"Test R2 Score: {ridge_test_score:.4f}")
    print(f"MAE: {ridge_mae:.4f}")
    print(f"RMSE: {ridge_rmse:.4f}")
    
    # 회귀 계수 데이터프레임 생성
    coef_df = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': model.coef_})
    return coef_df,test_score, y_pred
    
# =====================================================================
# 2. 실행
# =====================================================================

# 이코노미 데이터 분석 
eco_coef, eco_r2, eco_pred = analyze_ridge_log_scale(X_train_eco, X_test_eco, y_train_eco, y_test_eco, "Economy")

# 비즈니스 데이터 분석 
bus_coef, bus_r2, bus_pred = analyze_ridge_log_scale(X_train_bus, X_test_bus, y_train_bus, y_test_bus, "Business")

# =====================================================================
# 3. 영향력(Coefficient) 상위 5개 변수 추출
# =====================================================================

print("\n[Economy: 가격 상승 기여도 Top 5]")
print(eco_coef.sort_values(by='Coefficient', ascending=False).head(5))

print("\n[Business: 가격 상승 기여도 Top 5]")
print(bus_coef.sort_values(by='Coefficient', ascending=False).head(5))


In [7]:
############################################## 
# 2. Ridge 시각화 (predicted vs actual)
############################################## 

In [20]:
############################################## 
# 3. XGBoost
############################################## 
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

# 1. XGBoost용 파라미터 그리드 설정 
xgb_param_grid = {
    'n_estimators': [500, 1000],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

# 2. 모델 최적화 및 분석 함수 정의 
def analyze_xgb_model(X_train, X_test, y_train, y_test, label):
    
    xgb_model = XGBRegressor(verbosity=0, random_state=42, n_jobs=-1, tree_method='hist')

    # GridSearchCV 설정
    grid_search = GridSearchCV(
        estimator=xgb_model, 
        param_grid=xgb_param_grid, 
        cv=3, 
        scoring='r2', 
        verbose=1 
    )
    
    print(f"\n--- [{label}] XGBoost 결과 ---")
    
    grid_search.fit(X_train, y_train)
    xgb_best_model = grid_search.best_estimator_
    xgb_y_pred = xgb_best_model.predict(X_test)

    # 예측 및 지표 계산
    xgb_train_score = xgb_best_model.score(X_train, y_train)
    xgb_test_score = xgb_best_model.score(X_test, y_test)
    xgb_mae = mean_absolute_error(y_test, xgb_y_pred)
    xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_y_pred))
    
    print(f"최적 파라미터: {grid_search.best_params_}")
    print(f"Train R2 Score: {xgb_train_score:.4f}") 
    print(f"Test R2 Score: {xgb_test_score:.4f}")
    print(f"MAE: {xgb_mae:.4f}")
    print(f"RMSE: {xgb_rmse:.4f}")

    # 중요도 데이터프레임 생성 
    xgb_imp_df = pd.DataFrame({
        'Feature': X_train.columns, 
        'Importance': xgb_best_model.feature_importances_  
    })
    
    return xgb_best_model, xgb_imp_df

# =====================================================================
# 2. 실행
# =====================================================================

# 이코노미 데이터 분석 
best_xgb_eco, eco_imp_df = analyze_xgb_model(
    X_train_eco, X_test_eco, y_train_eco, y_test_eco, "Economy"
)

# 비지니스 데이터 분석 
best_xgb_bus, bus_imp_df = analyze_xgb_model(
    X_train_bus, X_test_bus, y_train_bus, y_test_bus, "Business"
)

# =====================================================================
# 3. 중요도 상위 5개 변수 비교
# =====================================================================
print("\n[Economy: XGBoost 중요도 Top 5]")
print(eco_imp_df.sort_values(by='Importance', ascending=False).head(5))

print("\n[Business: XGBoost 중요도 Top 5]")
print(bus_imp_df.sort_values(by='Importance', ascending=False).head(5))


--- [Economy] XGBoost 결과 ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
최적 파라미터: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 1000, 'subsample': 0.8}
Train R2 Score: 0.9109
Test R2 Score: 0.8961
MAE: 0.1149
RMSE: 0.1693

--- [Business] XGBoost 결과 ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
최적 파라미터: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 1000, 'subsample': 0.8}
Train R2 Score: 0.9473
Test R2 Score: 0.9321
MAE: 0.0463
RMSE: 0.0733

[Economy: XGBoost 중요도 Top 5]
                      Feature  Importance
49  booking_period_22일 이상(장기)    0.137956
8             airline_Vistara    0.064319
3                   days_left    0.059764
4           airline_Air_India    0.045418
5            airline_GO_FIRST    0.044130

[Business: XGBoost 중요도 Top 5]
                   Feature  Importance
0                    stops    0.064738
2                 duration    0.062110
4        airline_Air_In

In [ ]:
############################################## 
# 4. XGBoost 시각화 (predicted vs actual)
# 이코노미와 비지니스의 train, test score 비교 후 인사이트 도출 
############################################## 

In [22]:
############################################## 
# 5. LightGBM
############################################## 

from lightgbm import LGBMRegressor

# 1. LightGBM용 파라미터 그리드 설정
lgb_param_grid = {
    'n_estimators': [500, 1000],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8],
    'random_state': [42],
    'force_col_wise': [True], 
    'verbose': [-1]
}

# 2. 모델 최적화 및 분석 함수 정의 
def run_lgb_tuning(X_train, X_test, y_train, y_test, label):
    
    lgb_model = LGBMRegressor(n_jobs=-1)

    # GridSearchCV 설정
    lgb_grid_search = GridSearchCV(
        estimator=lgb_model, 
        param_grid=lgb_param_grid, 
        cv=3, 
        scoring='r2', 
        verbose=1
    )

    print(f"\n--- [{label}] LightGBM 최적화 시작 ---")
    lgb_grid_search.fit(X_train, y_train)

    # 최적 모델 및 파라미터 저장
    lgb_best_model = lgb_grid_search.best_estimator_
    lgb_best_params = lgb_grid_search.best_params_

    # 예측 및 지표 계산
    lgb_y_pred = lgb_best_model.predict(X_test)
    
    lgb_train_score = lgb_best_model.score(X_train, y_train)
    lgb_test_score = lgb_best_model.score(X_test, y_test)  
    lgb_mae = mean_absolute_error(y_test, lgb_y_pred)
    lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_y_pred))
    
    print(f"최적 파라미터: {lgb_grid_search.best_params_}")
    print(f"Train R2 Score: {lgb_train_score:.4f}") 
    print(f"Test R2 Score: {lgb_test_score:.4f}")
    print(f"MAE: {lgb_mae:.4f}")
    print(f"RMSE: {lgb_rmse:.4f}")

    # 중요도 데이터프레임 생성
    lgb_imp_df = pd.DataFrame({
        'Feature': X_train.columns, 
        'Importance': lgb_best_model.feature_importances_
    })
    
    return lgb_best_model, lgb_imp_df

# =====================================================================
# 2. 실행
# =====================================================================

# 이코노미 데이터 분석 
best_lgb_eco, eco_lgb_imp = run_lgb_tuning(
    X_train_eco, X_test_eco, y_train_eco, y_test_eco, "Economy"
)

# 비지니스 데이터 분석 
best_lgb_bus, bus_lgb_imp = run_lgb_tuning(
    X_train_bus, X_test_bus, y_train_bus, y_test_bus, "Business"
)

# =====================================================================
# 3. 중요도 상위 5개 변수 비교
# =====================================================================
print("\n[Economy: LightGBM 중요도 Top 5]")
print(eco_lgb_imp.sort_values(by='Importance', ascending=False).head(5))

print("\n[Business: LightGBM 중요도 Top 5]")
print(bus_lgb_imp.sort_values(by='Importance', ascending=False).head(5))


--- [Economy] LightGBM 최적화 시작 ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
최적 파라미터: {'force_col_wise': True, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 1000, 'random_state': 42, 'subsample': 0.8, 'verbose': -1}
Train R2 Score: 0.8898
Test R2 Score: 0.8814
MAE: 0.1243
RMSE: 0.1809

--- [Business] LightGBM 최적화 시작 ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
최적 파라미터: {'force_col_wise': True, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 1000, 'random_state': 42, 'subsample': 0.8, 'verbose': -1}
Train R2 Score: 0.9286
Test R2 Score: 0.9212
MAE: 0.0504
RMSE: 0.0789

[Economy: LightGBM 중요도 Top 5]
             Feature  Importance
2           duration        8456
3          days_left        5365
0              stops        1005
6     airline_Indigo         972
4  airline_Air_India         838

[Business: LightGBM 중요도 Top 5]
                   Feature  Importance
2                 duration       12496
3                days_left        39

In [ ]:
############################################## 
# 4. XGBoost 시각화 (predicted vs actual)
# 이코노미와 비지니스의 train, test score 비교 후 인사이트 도출 (가장 적합한 모델 설명)
# ridge,xgboost,lightgbm에서 추출한 각모델의 중요도 비교 후 인사이트 도출 (왜 다르게 나왔는지도 분석)
############################################## 

In [23]:
############################################## 
# 4. Stacking (릿지 -> 랜포로 변경)
############################################## 
from sklearn.ensemble import StackingRegressor
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
import numpy as np

def run_final_stacking(X_train, X_test, y_train, y_test, label):
    # 1. Base Learners

    base_models = [
        ('ridge', make_pipeline(StandardScaler(), Ridge(alpha=10.0))),
        ('xgb', XGBRegressor(
            n_estimators=500, max_depth=6, learning_rate=0.05, 
            subsample=0.8, colsample_bytree=0.8, 
            random_state=42, tree_method='hist', n_jobs=-1
        )),
        ('lgb', LGBMRegressor(
            n_estimators=500, max_depth=6, learning_rate=0.05, 
            subsample=0.8, random_state=42, 
            force_col_wise=True, verbose=-1, n_jobs=-1
        ))
    ]

    # 2. Meta Learner (과적합 방지를 위해 Ridge)

    meta_model = Ridge(alpha=1.0)

    # 3. 스태킹 모델 정의
    
    stacking_reg = StackingRegressor(
        estimators=base_models,
        final_estimator=meta_model,
        cv=5, 
        n_jobs=-1,
        passthrough=False # 기반 모델의 예측값만 사용할 경우 False
    )

    print(f"\n--- [{label}] 최종 스태킹 모델 결과")
    stacking_reg.fit(X_train, y_train)

    # 4. 성능 평가
    
    y_pred = stacking_reg.predict(X_test)
    
    stacking_train_score = stacking_reg.score(X_train, y_train)
    stacking_test_score = stacking_reg.score(X_test, y_test)
    stacking_mae = mean_absolute_error(y_test, y_pred)
    stacking_rmse = np.sqrt(mean_squared_error(y_test, y_pred))    

    print(f"Train R2 Score: {stacking_train_score:.4f}") 
    print(f"Test R2 Score: {stacking_test_score:.4f}")
    print(f"MAE: {stacking_mae :.4f}")
    print(f"RMSE: {stacking_rmse:.4f}")


    return stacking_reg, y_pred

# =====================================================================
# 2. 실행
# =====================================================================

# 이코노미 데이터 분석 
stacking_model_eco, pred_eco = run_final_stacking(
    X_train_eco, X_test_eco, y_train_eco, y_test_eco, "Economy"
)

# 비지니스 스태킹 실행
stacking_model_bus, pred_bus = run_final_stacking(
    X_train_bus, X_test_bus, y_train_bus, y_test_bus, "Business"
)



--- [Economy] 최종 스태킹 모델 결과
Train R2 Score: 0.8652
Test R2 Score: 0.8580
MAE: 0.1390
RMSE: 0.1980

--- [Business] 최종 스태킹 모델 결과
Train R2 Score: 0.8971
Test R2 Score: 0.8899
MAE: 0.0651
RMSE: 0.0933
